<a href="https://colab.research.google.com/github/sjsu-cs131-spring26/boxscorelab-team9-sportanalytics/blob/sprint5_minn/docs/meetings/ProjAssign5_minn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


In [2]:
!ls "/content/drive/MyDrive/cs131_Team09_sec1"

Games.csv  TeamStatistics.csv


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sprint5_minn")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark version: 4.0.2
Shuffle partitions: 8


In [4]:
#Task1: Load Games.csv into PySpark and identify the join key and useful context columns.
DATA_DIR = "/content/drive/MyDrive/cs131_Team09_sec1"

games = spark.read.csv(f"{DATA_DIR}/Games.csv", header=True, inferSchema=True)

print("Row count:", games.count())
print("Column count:", len(games.columns))
print()
games.printSchema()
print()
games.show(5, truncate=False)

Row count: 72921
Column count: 22

root
 |-- gameId: integer (nullable = true)
 |-- gameDateTimeEst: timestamp (nullable = true)
 |-- hometeamCity: string (nullable = true)
 |-- hometeamName: string (nullable = true)
 |-- hometeamId: integer (nullable = true)
 |-- awayteamCity: string (nullable = true)
 |-- awayteamName: string (nullable = true)
 |-- awayteamId: integer (nullable = true)
 |-- homeScore: integer (nullable = true)
 |-- awayScore: integer (nullable = true)
 |-- winner: integer (nullable = true)
 |-- gameType: string (nullable = true)
 |-- gameSubtype: string (nullable = true)
 |-- gameLabel: string (nullable = true)
 |-- gameSubLabel: string (nullable = true)
 |-- seriesGameNumber: double (nullable = true)
 |-- attendance: integer (nullable = true)
 |-- arenaId: integer (nullable = true)
 |-- arenaName: string (nullable = true)
 |-- arenaCity: string (nullable = true)
 |-- arenaState: string (nullable = true)
 |-- officials: string (nullable = true)


+--------+----------

In [5]:
team_stats = spark.read.csv(f"{DATA_DIR}/TeamStatistics.csv", header=True, inferSchema=True)

print("Row count:", team_stats.count())
print("Column count:", len(team_stats.columns))
print()
team_stats.printSchema()
print()
team_stats.show(5, truncate=False)

Row count: 145844
Column count: 48

root
 |-- gameId: integer (nullable = true)
 |-- gameDateTimeEst: timestamp (nullable = true)
 |-- teamCity: string (nullable = true)
 |-- teamName: string (nullable = true)
 |-- teamId: integer (nullable = true)
 |-- opponentTeamCity: string (nullable = true)
 |-- opponentTeamName: string (nullable = true)
 |-- opponentTeamId: integer (nullable = true)
 |-- home: integer (nullable = true)
 |-- win: integer (nullable = true)
 |-- teamScore: integer (nullable = true)
 |-- opponentScore: integer (nullable = true)
 |-- assists: integer (nullable = true)
 |-- blocks: integer (nullable = true)
 |-- steals: integer (nullable = true)
 |-- fieldGoalsAttempted: integer (nullable = true)
 |-- fieldGoalsMade: integer (nullable = true)
 |-- fieldGoalsPercentage: double (nullable = true)
 |-- threePointersAttempted: integer (nullable = true)
 |-- threePointersMade: integer (nullable = true)
 |-- threePointersPercentage: double (nullable = true)
 |-- freeThrowsAtt

In [6]:
#Task 2: Join TeamStatistics.csv DataFrame with Games.csv on gameId
from pyspark.sql import functions as F

# Keeps the joined DataFrame slim and avoids duplicating gameDateTimeEst.
games_slim = games.select(
    "gameId",
    "gameType",
    "hometeamId",
    "awayteamId",
    "winner"
)

# Inner join on gameId. Using the string form of the key name
# automatically de-duplicate the join column.
joined = team_stats.join(games_slim, on="gameId", how="inner")

# Derive season from gameDateTimeEst.
# Convention: label by the ENDING year (2025-26 season -> 2026).
# If the game's month >= October, the season ends next year.
joined = joined.withColumn(
    "season",
    F.when(F.month("gameDateTimeEst") >= 10, F.year("gameDateTimeEst") + 1)
     .otherwise(F.year("gameDateTimeEst"))
)

# Derive home_away_flag by comparing this row's teamId to Games' home/away teamIds
joined = joined.withColumn(
    "home_away_flag",
    F.when(F.col("teamId") == F.col("hometeamId"), "home")
     .when(F.col("teamId") == F.col("awayteamId"), "away")
     .otherwise(F.lit(None))
)

print("TeamStatistics rows before join:", team_stats.count())
print("Joined rows after join:          ", joined.count())
print()

# Spot-check: show a few rows with derived columns side-by-side
# with the existing `home` column (which will be used as a cross-check in Task 3)
joined.select(
    "gameId", "teamName", "teamId",
    "hometeamId", "awayteamId",
    "home_away_flag", "home",
    "season", "gameType"
).show(5, truncate=False)

TeamStatistics rows before join: 145844
Joined rows after join:           145842

+--------+--------+----------+----------+----------+--------------+----+------+--------------+
|gameId  |teamName|teamId    |hometeamId|awayteamId|home_away_flag|home|season|gameType      |
+--------+--------+----------+----------+----------+--------------+----+------+--------------+
|22500935|Clippers|1610612746|1610612746|1610612752|home          |1   |2026  |Regular Season|
|22500935|Knicks  |1610612752|1610612746|1610612752|away          |0   |2026  |Regular Season|
|22500934|Warriors|1610612744|1610612762|1610612744|away          |0   |2026  |Regular Season|
|22500934|Jazz    |1610612762|1610612762|1610612744|home          |1   |2026  |Regular Season|
|22500933|Nuggets |1610612743|1610612760|1610612743|away          |0   |2026  |Regular Season|
+--------+--------+----------+----------+----------+--------------+----+------+--------------+
only showing top 5 rows


In [7]:
from pyspark.sql import functions as F

# Task 3 Quality Check
joined.cache()

# --- Check 1: Row count before vs after join ---
print("=" * 60)
print("Check 1: Row counts before/after join")
print("=" * 60)
rows_before = team_stats.count()
rows_after = joined.count()
print(f"TeamStatistics rows (before join): {rows_before:,}")
print(f"Joined rows (after join):          {rows_after:,}")
print(f"Rows dropped in join:              {rows_before - rows_after}")

# Investigate the dropped rows — gameIds in TeamStatistics but NOT in Games
orphans = team_stats.join(games.select("gameId"), on="gameId", how="left_anti")
print(f"\nOrphan rows (in TeamStatistics but no matching gameId in Games): {orphans.count()}")
orphans.select("gameId", "gameDateTimeEst", "teamName").show(truncate=False)

# --- Check 2: Null counts in key analysis columns ---
print("=" * 60)
print("Check 2: Null counts in key columns")
print("=" * 60)
important_cols = [
    "gameId", "teamId", "teamName", "season", "home_away_flag",
    "teamScore", "fieldGoalsAttempted", "assists", "turnovers"
]
null_counts = joined.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in important_cols
])
null_counts.show(truncate=False)

# --- Check 3: Duplicate (gameId, teamName) combinations ---
print("=" * 60)
print("Check 3: Duplicate (gameId, teamName) rows")
print("=" * 60)
dupes = joined.groupBy("gameId", "teamName").count().filter("count > 1")
dupe_count = dupes.count()
print(f"Duplicate (gameId, teamName) combinations: {dupe_count}")
if dupe_count > 0:
    dupes.show(10)

# --- Check 4: Missing home/away assignments + cross-check with existing `home` ---
print("=" * 60)
print("Check 4: Home/away flag validation")
print("=" * 60)
missing_flag = joined.filter(F.col("home_away_flag").isNull()).count()
print(f"Rows with null home_away_flag: {missing_flag}")

disagreements = joined.filter(
    ((F.col("home_away_flag") == "home") & (F.col("home") == 0)) |
    ((F.col("home_away_flag") == "away") & (F.col("home") == 1))
).count()
print(f"Rows where derived flag disagrees with existing `home` column: {disagreements}")

# --- Check 5: Consistency — does win align with score? ---
print("=" * 60)
print("Check 5: Does `win` align with teamScore > opponentScore?")
print("=" * 60)
win_check = joined.withColumn(
    "win_aligns",
    (F.col("win") == 1) == (F.col("teamScore") > F.col("opponentScore"))
)
win_check.groupBy("win_aligns").count().show()

# --- Check 6: Manual spot-check — 5 random joined rows ---
print("=" * 60)
print("Check 6: Manual spot-check — 5 random joined rows")
print("=" * 60)
joined.select(
    "gameId", "season", "teamName", "home_away_flag", "home",
    "teamScore", "opponentScore", "win", "gameType"
).orderBy(F.rand(seed=42)).limit(5).show(truncate=False)

Check 1: Row counts before/after join
TeamStatistics rows (before join): 145,844
Joined rows (after join):          145,842
Rows dropped in join:              2

Orphan rows (in TeamStatistics but no matching gameId in Games): 2
+--------+-------------------+--------+
|gameId  |gameDateTimeEst    |teamName|
+--------+-------------------+--------+
|22500652|2026-01-25 14:00:00|NULL    |
|22500651|2026-01-25 10:30:00|NULL    |
+--------+-------------------+--------+

Check 2: Null counts in key columns
+------+------+--------+------+--------------+---------+-------------------+-------+---------+
|gameId|teamId|teamName|season|home_away_flag|teamScore|fieldGoalsAttempted|assists|turnovers|
+------+------+--------+------+--------------+---------+-------------------+-------+---------+
|0     |0     |0       |0     |0             |0        |34306              |35087  |40815    |
+------+------+--------+------+--------------+---------+-------------------+-------+---------+

Check 3: Duplicate

In [8]:
# When are the nulls concentrated? Look by season
print("=" * 60)
print("Nulls in key stats, broken down by season (only showing seasons with nulls)")
print("=" * 60)
nulls_by_season = joined.groupBy("season").agg(
    F.count("*").alias("rows"),
    F.sum(F.when(F.col("fieldGoalsAttempted").isNull(), 1).otherwise(0)).alias("null_FGA"),
    F.sum(F.when(F.col("assists").isNull(), 1).otherwise(0)).alias("null_ast"),
    F.sum(F.when(F.col("turnovers").isNull(), 1).otherwise(0)).alias("null_tov")
).filter(
    (F.col("null_FGA") > 0) | (F.col("null_ast") > 0) | (F.col("null_tov") > 0)
).orderBy("season")

nulls_by_season.show(100, truncate=False)

# The 4 weird win-alignment rows
print("=" * 60)
print("Rows where `win` disagrees with teamScore > opponentScore")
print("=" * 60)
joined.filter(
    (F.col("win") == 1) != (F.col("teamScore") > F.col("opponentScore"))
).select(
    "gameId", "season", "teamName", "teamScore", "opponentScore",
    "win", "gameType"
).show(truncate=False)

Nulls in key stats, broken down by season (only showing seasons with nulls)
+------+----+--------+--------+--------+
|season|rows|null_FGA|null_ast|null_tov|
+------+----+--------+--------+--------+
|1947  |40  |40      |40      |40      |
|1948  |48  |42      |42      |48      |
|1949  |166 |166     |166     |166     |
|1950  |274 |270     |274     |274     |
|1951  |442 |410     |412     |442     |
|1952  |454 |392     |448     |454     |
|1953  |470 |408     |470     |470     |
|1954  |550 |432     |532     |550     |
|1955  |618 |551     |614     |618     |
|1956  |622 |572     |622     |622     |
|1957  |614 |590     |612     |614     |
|1958  |618 |592     |618     |618     |
|1959  |620 |580     |616     |620     |
|1960  |650 |552     |649     |650     |
|1961  |682 |489     |598     |660     |
|1962  |778 |710     |774     |778     |
|1963  |778 |760     |776     |778     |
|1964  |774 |736     |754     |774     |
|1965  |772 |772     |772     |772     |
|1966  |774 |756     |

From 1947-1984 there are a lot of null values for turnovers, FGA and assists, which reflects that NBA started consistently tracking all advanced stats. So I will be flitering seasons years to calculate effiency to seasons >= 1986.

Also I found out there are 4 rows of data anomalies which disgrees with teamScore > opponentScore which means the matches were either tied(data errors, OT score wasn't captured) or cancelled.

In [9]:
#Task 4 Use a Spark window function to rank teams by offensive efficiency within each season to create a season-by-season ranking of team offensive efficiency
from pyspark.sql import Window
from pyspark.sql import functions as F

# Apply all filters from our data quality findings
filtered = joined.filter(
    (F.col("season") >= 1986) &
    (F.col("gameType") == "Regular Season") &
    (F.col("teamScore") != F.col("opponentScore"))
)

# Aggregate at the season-team level
# Group by season + teamId (stable) + teamName (display), compute per-game averages.
team_season = filtered.groupBy("season", "teamId", "teamName").agg(
    F.count("*").alias("n_games"),
    F.round(F.avg("teamScore"), 2).alias("avgScore"),
    F.round(F.avg("fieldGoalsAttempted"), 2).alias("avgFGA"),
    F.round(F.avg("assists"), 2).alias("avgAssists"),
    F.round(F.avg("turnovers"), 2).alias("avgTurnovers")
)

# Compute efficiency = avgScore / avgFGA
team_season = team_season.withColumn(
    "efficiency",
    F.round(F.col("avgScore") / F.col("avgFGA"), 4)
)

# Window function — rank teams within each season
# partitionBy(season) means "reset the ranking at each season boundary"
# orderBy(desc(efficiency)) means "higher efficiency = better rank"
win_spec = Window.partitionBy("season").orderBy(F.desc("efficiency"))
team_season_ranked = team_season.withColumn("rank", F.rank().over(win_spec))

# Arrange the final columns and sort for readability
team_season_ranked = team_season_ranked.select(
    "season", "teamName", "efficiency", "rank",
    "avgScore", "avgAssists", "avgTurnovers", "n_games"
).orderBy("season", "rank")

team_season_ranked.cache()

# --- Sanity checks ---
print("Total team-season rows:", team_season_ranked.count())
print("Distinct seasons:       ", team_season_ranked.select("season").distinct().count())
print()

print("Top 5 of the 2025-26 season (season=2026):")
team_season_ranked.filter(F.col("season") == 2026).show(5, truncate=False)

print("Top 5 of the 1986 season (oldest season in our filter):")
team_season_ranked.filter(F.col("season") == 1986).show(5, truncate=False)

Total team-season rows: 1177
Distinct seasons:        41

Top 5 of the 2025-26 season (season=2026):
+------+------------+----------+----+--------+----------+------------+-------+
|season|teamName    |efficiency|rank|avgScore|avgAssists|avgTurnovers|n_games|
+------+------------+----------+----+--------+----------+------------+-------+
|2026  |Nuggets     |1.3847    |1   |120.26  |28.02     |12.29       |65     |
|2026  |Lakers      |1.3823    |2   |115.88  |25.42     |14.05       |64     |
|2026  |Clippers    |1.3608    |3   |112.8   |23.41     |13.88       |64     |
|2026  |Timberwolves|1.3398    |4   |118.61  |26.19     |13.97       |64     |
|2026  |Thunder     |1.3364    |5   |118.79  |25.48     |11.97       |66     |
+------+------------+----------+----+--------+----------+------------+-------+
only showing top 5 rows
Top 5 of the 1986 season (oldest season in our filter):
+------+-------------+----------+----+--------+----------+------------+-------+
|season|teamName     |effici

In [10]:
# Task 5: Top 10 team-seasons by offensive efficiency

top10 = (
    team_season_ranked
    .orderBy(F.desc("efficiency"))
    .limit(10)
    .withColumnRenamed("rank", "season_rank")
    .select(
        "season", "teamName", "efficiency", "season_rank",
        "avgScore", "avgAssists", "avgTurnovers", "n_games"
    )
)

print("TOP 10 TEAM-SEASONS BY OFFENSIVE EFFICIENCY (1986-2026, Regular Season)")
print("=" * 80)
top10.show(10, truncate=False)

TOP 10 TEAM-SEASONS BY OFFENSIVE EFFICIENCY (1986-2026, Regular Season)
+------+---------+----------+-----------+--------+----------+------------+-------+
|season|teamName |efficiency|season_rank|avgScore|avgAssists|avgTurnovers|n_games|
+------+---------+----------+-----------+--------+----------+------------+-------+
|2026  |Nuggets  |1.3847    |1          |120.26  |28.02     |12.29       |65     |
|2026  |Lakers   |1.3823    |2          |115.88  |25.42     |14.05       |64     |
|1995  |Jazz     |1.3766    |1          |106.41  |27.51     |15.18       |82     |
|2023  |76ers    |1.3753    |1          |115.22  |25.16     |13.67       |82     |
|2023  |Kings    |1.3686    |2          |120.71  |27.28     |13.49       |82     |
|2026  |Clippers |1.3608    |3          |112.8   |23.41     |13.88       |64     |
|1997  |Jazz     |1.3598    |1          |103.1   |26.82     |15.35       |82     |
|2021  |Nets     |1.3574    |1          |118.57  |26.79     |13.54       |72     |
|2023  |Maveric

In [11]:
top10.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{DATA_DIR}/top10_csv")